# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)


## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```


## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$


## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [ ]:
import numpy as np

UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)

# Relativo a la dirección de movimiento intentada: a qué acción
# corresponde una desviación hacia la izquierda / derecha del robot.
LEFT_OF = {UP: LEFT, DOWN: RIGHT, LEFT: DOWN, RIGHT: UP}
RIGHT_OF = {UP: RIGHT, DOWN: LEFT, LEFT: UP, RIGHT: DOWN}


class WarehouseMDP:
    def __init__(self, living_reward=-1.0, gamma=0.9, slip_intended=0.60, slip_dev=0.20):
        self.height = 5
        self.width = 6

        self.start = (0, 0)

        # Estanterías (paredes)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Piso resbaloso
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # Estados terminales: (row, col) -> recompensa
        self.terminal_states = {
            (0, 5): 10,   # entrega
            (2, 2): 2,    # carga
            (3, 5): -10,  # peligro mortal
        }

        # Peligros no terminales
        self.danger_states = {
            (1, 4): -3,
            (4, 1): -3,
        }

        self.living_reward = living_reward
        self.gamma = gamma

        # Probabilidades del piso resbaloso (parametrizadas para los experimentos)
        self.slip_intended = slip_intended
        self.slip_dev = slip_dev

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if not (0 <= r < self.height and 0 <= c < self.width):
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if self.is_valid_state((r, c))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # Un estado terminal no tiene dinámica: se queda en sí mismo.
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_intended, p_dev = self.slip_intended, self.slip_dev
        else:
            p_intended, p_dev = 0.90, 0.05

        left_action = LEFT_OF[action]
        right_action = RIGHT_OF[action]

        outcomes = [
            (action, p_intended),
            (left_action, p_dev),
            (right_action, p_dev),
        ]

        probs = {}
        for a, p in outcomes:
            dr, dc = a
            next_state = (state[0] + dr, state[1] + dc)
            if not self.is_valid_state(next_state):
                next_state = state
            probs[next_state] = probs.get(next_state, 0.0) + p

        return list(probs.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [ ]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    total = 0.0
    for next_state, prob in grid.get_transition_probs(state, action):
        total += prob * V[next_state]
    return total


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for i in range(max_iter):
        V_new = {}
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                q_values = [
                    grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
                    for a in grid.actions
                ]
                V_new[s] = max(q_values)

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < threshold:
            return V, i + 1

    return V, max_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}

    for s in grid.states():
        if grid.is_terminal(s):
            policy[s] = None
            continue

        best_action, best_q = None, float("-inf")
        for a in grid.actions:
            q = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
            if q > best_q:
                best_q, best_action = q, a

        policy[s] = best_action

    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for i in range(max_iter):
        V_new = {}
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                a = policy[s]
                V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < threshold:
            break

    return V


def policy_improvement(grid, V):
    new_policy = {}

    for s in grid.states():
        if grid.is_terminal(s):
            new_policy[s] = None
            continue

        best_action, best_q = None, float("-inf")
        for a in grid.actions:
            q = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
            if q > best_q:
                best_q, best_action = q, a

        new_policy[s] = best_action

    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria
    policy = {
        s: (grid.actions[0] if not grid.is_terminal(s) else None)
        for s in grid.states()
    }

    history = []
    V = {s: 0.0 for s in grid.states()}

    for i in range(max_iter):
        # 2. evaluación
        V = policy_evaluation(grid, policy, threshold=threshold)

        # 3. mejora
        new_policy = policy_improvement(grid, V)

        changed = sum(
            1 for s in grid.states()
            if not grid.is_terminal(s) and new_policy[s] != policy[s]
        )
        history.append(changed)

        policy = new_policy

        # 4. repetir hasta estabilidad
        if changed == 0:
            break

    return policy, V, history


## Parte 4 — Visualización y comparación

In [ ]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?


### Respuestas (1–5)

1. Con los parámetros base (`living_reward=-1.0`, `γ=0.9`), el robot **prefiere la carga +2**. La política óptima desde `START` sigue la ruta `(0,0) → (0,1) → (0,2) → (1,2) → (2,2)` (4 pasos), con `V(START) = -2.575` (ver celda de ejecución de Parte 4).

2. Porque cada paso cuesta `-1` y además está descontado por `γ`. El camino hacia la entrega `+10` es más largo y pasa cerca de celdas de peligro `-3` y de una celda resbalosa junto al peligro mortal `-10`. El valor esperado de arriesgarse a recorrer ese camino, descontado, termina siendo menor que llegar rápido a una recompensa más chica pero seR. En otras palabras: `γ` y el costo por paso hacen que "cerca y seguro" le gane a "lejos y grande".

3. En `(1,2)` y `(2,1)` (las celdas resbalosas del camino relevante): ahí la probabilidad de desviación sube de 5%/5% (piso normal) a 20%/20%, lo que aumenta el riesgo de terminar en un estado no deseado cerca de las zonas de peligro. Eso hace que esas rutas valgan menos de lo que un cálculo determinista (sin incertidumbre) sugeriría.

4. El costo por paso `-1` es lo que introduce la "urgencia": sin él, no habría penalización por tomar caminos más largos y el agente siempre preferiría la recompensa terminal más grande, sin importar la distancia. Es lo que hace que la distancia (número de pasos esperado) entre en la decisión.

5. Porque la dinámica `T(s,a,s')` depende de si el estado de origen `s` es piso normal o piso resbaloso — son dos distribuciones de probabilidad distintas (0.90/0.05/0.05 vs. 0.60/0.20/0.20). Si se usara la misma `T` para todos los estados, se perdería esa diferencia y el modelo no reflejaría el riesgo real de atravesar una celda resbalosa.

---

### Experimento A — Menos costo por paso

Cambia `living_reward = -0.1` y **antes de ejecutar**, predicción: al bajar casi a cero el costo de moverse, ya no hay urgencia por terminar rápido, así que debería preferir la recompensa más grande (`entrega +10`) aunque el camino sea más largo.


In [ ]:
grid_A = WarehouseMDP(living_reward=-0.1)

V_A, n_A = value_iteration(grid_A)
pi_A = extract_policy(grid_A, V_A)

print("=== EXPERIMENTO A: living_reward = -0.1 ===")
print("V(START) =", round(V_A[grid_A.start], 3), " (baseline era -2.575)")
print("\nPolítica:")
print_policy(grid_A, pi_A)


**Resultado A:** se confirma la predicción. `V(START)` sube de `-2.575` a `+1.649`, y la política desde `START` ahora sí va hacia la **entrega +10**, siguiendo `(0,0) → (0,1) → (0,2) → (1,2) → (1,3) → (1,4) → (1,5) → (0,5)`. Al costar casi nada moverse, vale la pena recorrer el camino largo por la recompensa mayor — confirma la respuesta de la pregunta 4.

---

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso: `0.60 → 0.40`, repartiendo el restante entre las dos desviaciones (0.30 / 0.30).


In [ ]:
grid_B = WarehouseMDP(slip_intended=0.40, slip_dev=0.30)

V_B, n_B = value_iteration(grid_B)
pi_B = extract_policy(grid_B, V_B)

print("=== EXPERIMENTO B: piso resbaloso 0.40 / 0.30 / 0.30 ===")
print("V(START) =", round(V_B[grid_B.start], 3), " (baseline era -2.575)")
print("\nPolítica:")
print_policy(grid_B, pi_B)


**Resultado B:** la política **no cambia** — sigue yendo a carga `+2` por la misma ruta. Pero `V(START)` empeora levemente: de `-2.575` a `-2.706`. Tiene sentido: más incertidumbre en el piso resbaloso reduce el valor esperado del camino (más riesgo de terminar donde no se quiere), pero no alcanza a cambiar cuál es la mejor decisión porque la carga sigue siendo la opción más cercana y segura.

---

### Experimento C — Más paciencia

Cambia `gamma = 0.99`. Pregunta: ¿la política valora más la recompensa `+10` distante?


In [ ]:
grid_C = WarehouseMDP(gamma=0.99)

V_C, n_C = value_iteration(grid_C)
pi_C = extract_policy(grid_C, V_C)

print("=== EXPERIMENTO C: gamma = 0.99 ===")
print("V(START) =", round(V_C[grid_C.start], 3), " (baseline era -2.575)")
print("\nPolítica:")
print_policy(grid_C, pi_C)


**Resultado C:** sí. Con `γ=0.99` la política cambia hacia la **entrega +10**, igual que en el experimento A, y `V(START)` mejora de `-2.575` a `-1.456`. Con menos descuento (más "paciencia"), las recompensas lejanas pierden menos valor por estar lejos, así que la recompensa grande vuelve a ser la más atractiva aunque el costo por paso siga siendo `-1`.

---

### Bonus

Barrido de `living_reward` para encontrar el punto donde la política desde `START` cambia entre ir a carga `+2` e intentar llegar a entrega `+10`.


In [ ]:
prev_target = None
for i in range(-100, 1):
    lr = i / 100
    g = WarehouseMDP(living_reward=lr)
    V, _ = value_iteration(g)
    pi = extract_policy(g, V)

    # Recorrido "intencional" desde START siguiendo la política
    s = g.start
    for _ in range(30):
        if g.is_terminal(s):
            break
        a = pi[s]
        ns = (s[0] + a[0], s[1] + a[1])
        s = ns if g.is_valid_state(ns) else s

    target = "ENTREGA(+10)" if s == (0, 5) else ("CARGA(+2)" if s == (2, 2) else str(s))
    if target != prev_target:
        print(f"living_reward = {lr:+.2f}  ->  destino desde START: {target}")
        prev_target = target


**Resultado del bonus:** el punto de cambio está entre `living_reward ≈ -0.80` y `living_reward ≈ -0.79`. Por debajo de ese valor (costo por paso más fuerte) la política prefiere la carga `+2`; por encima (costo por paso más suave) prefiere intentar llegar a la entrega `+10`.